# Milestone 3 — Text Vectorization & Embeddings

(a) Review rating band classification: TF-IDF baseline vs Keras embeddings

(b) Semantic product search with sentence embeddings

In [ ]:
import os
import sys
from pathlib import Path

for path in [
    Path("/content/smart-product-intelligence"),
    Path("/content/drive/MyDrive/smart-product-intelligence"),
    Path.cwd(),
    Path.cwd().parent,
]:
    if (path / "src" / "data.py").exists():
        os.chdir(path)
        if str(path) not in sys.path:
            sys.path.insert(0, str(path))
        print("Project root:", path)
        break

import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from sklearn.linear_model import LogisticRegression
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from src.data import artifacts_dir, load_splits
from src.models import build_embedding_text_classifier, set_global_seed
from src.utils import save_json, setup_notebook_path

setup_notebook_path()
set_global_seed(42)

In [ ]:
MAX_LEN = 128
MAX_WORDS = 10000
splits = load_splits()
train_r = splits["train_reviews"].copy()
val_r = splits["val_reviews"].copy()
test_r = splits["test_reviews"].copy()

for df in (train_r, val_r, test_r):
    df["text"] = (df["review_title"] + " " + df["review_text"]).str.strip()

y_train = train_r["rating_band"] - 1
y_val = val_r["rating_band"] - 1
y_test = test_r["rating_band"] - 1

In [ ]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=3)
X_train_tfidf = tfidf.fit_transform(train_r["text"])
X_val_tfidf = tfidf.transform(val_r["text"])
X_test_tfidf = tfidf.transform(test_r["text"])

bow_model = LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1)
bow_model.fit(X_train_tfidf, y_train)
bow_pred = bow_model.predict(X_test_tfidf)
bow_f1 = f1_score(y_test, bow_pred, average="macro")
print("TF-IDF macro-F1:", bow_f1)
print(classification_report(y_test, bow_pred))

In [ ]:
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<unk>")
tokenizer.fit_on_texts(train_r["text"])
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(train_r["text"]), maxlen=MAX_LEN)
X_val_seq = pad_sequences(tokenizer.texts_to_sequences(val_r["text"]), maxlen=MAX_LEN)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(test_r["text"]), maxlen=MAX_LEN)

emb_model = build_embedding_text_classifier(
    vocab_size=min(MAX_WORDS, len(tokenizer.word_index) + 1),
    num_classes=5,
    max_len=MAX_LEN,
)
emb_hist = emb_model.fit(
    X_train_seq,
    y_train,
    validation_data=(X_val_seq, y_val),
    epochs=5,
    batch_size=128,
    verbose=1,
)
emb_pred = np.argmax(emb_model.predict(X_test_seq, verbose=0), axis=1)
emb_f1 = f1_score(y_test, emb_pred, average="macro")
print("Embedding macro-F1:", emb_f1)

In [ ]:
from sentence_transformers import SentenceTransformer

products = splits["train_products"].copy()
products["full_text"] = (
    products["title"].fillna("") + " " + products["description"].fillna("")
).str.strip()

encoder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = encoder.encode(products["full_text"].tolist(), normalize_embeddings=True, show_progress_bar=True)

np.save(artifacts_dir() / "product_embeddings.npy", embeddings)
with open(artifacts_dir() / "product_embedding_index.json", "w", encoding="utf-8") as f:
    json.dump(products["product_id"].tolist(), f)

save_json({"tfidf_macro_f1": float(bow_f1), "embedding_macro_f1": float(emb_f1)}, artifacts_dir() / "text_metrics.json")

query = "hydrating face cream for dry skin"
q_vec = encoder.encode([query], normalize_embeddings=True)[0]
sims = embeddings @ q_vec
top_idx = np.argsort(-sims)[:5]
products.iloc[top_idx][["product_id", "title", "average_rating"]]